In [32]:
import xarray as xr
import numpy as np
import sys
import glob
import pandas as pd
import dask
import socket
import xesmf as xe
import warnings
warnings.filterwarnings('ignore')

### Set up dask cluster

In [3]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})
dask.config.set({"optimization.fuse.active": False})
dask.config.set({
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.8,
    "distributed.worker.memory.terminate": 0.95,
})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '/glade/derecho/scratch/islas/dask_tmp/',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='05:00:00',
    interface='mgt')

# scale up
cluster.scale(12)
#cluster.adapt(minimum=1, maximum=12)

# change your urls to the dask dashboard so that you can see it
hostname = socket.getfqdn()
dask.config.set({"distributed.dashboard.link":
        f"https://ondemand.hpc.ucar.edu/rnode/{hostname}/{{port}}/status"
})

client = Client(cluster)

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask_jobqueue/pbs.py:82: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/miniforge/envs/npl-2026a/l

In [35]:
cluster

PBSCluster(9aebf3ba, 'tcp://10.18.206.66:44103', workers=12, threads=12, memory=335.28 GiB)

In [6]:
# Location of ERA5 and output directory
basepath="/glade/campaign/collections/gdex/data/d633000/e5.oper.an.sfc/"
outpath="/glade/campaign/cgd/cas/islas/python_savs/rossbypalooza26/processing/ERA5/"

In [41]:
ystart=1993
yend=2025
var = "msl"
varinfile="MSL"
ourvar="slp"

### Output grid

In [11]:
grid_out = xr.Dataset({'lat':(['lat'], np.arange(-90,90,1))}, {'lon': (['lon'], np.arange(0,360,1))})

In [46]:
reusewgt=False
wgtfile=outpath+'wgtfile.nc'
alltimes=[]
alldat=[]
for iyear in range(ystart,yend+1,1):
    print(iyear)
    hindcast_months = pd.to_datetime([
        f"{iyear}-11-15",
        f"{iyear}-12-15",
        f"{iyear+1}-01-15",
        f"{iyear+1}-02-15",
        f"{iyear+1}-03-15"])

    allmonths = []
    for imon in hindcast_months:
        yearstr = str(imon.year)
        monstr = str(imon.month).zfill(2)
        file=basepath+"/"+yearstr+monstr+"/*_"+var+".*.nc"

        # read in the hourly data
        dat = xr.open_mfdataset(file, coords="minimal", join="override", 
                                decode_times=True, use_cftime=True, chunks={'time':-1})[varinfile]
        dat = dat.mean('time')

        # Regrid
        if (iyear == ystart) and (imon == hindcast_months[0]):
            regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=reusewgt, filename=wgtfile)
        dat_rg = regridder(dat)
        allmonths.append(dat_rg)
    allmonths = xr.concat(allmonths, dim='lead')
    allmonths['lead'] = np.arange(1,5+1,1)
    allmonths = allmonths.rename(ourvar)
    allmonths = allmonths.compute()
    alldat.append(allmonths)

    times = xr.DataArray(hindcast_months, dims=['lead'], coords=[np.arange(1,5+1,1)], name='time')
    
    alltimes.append(times)
alldat = xr.concat(alldat, dim='year')
alldat['year'] = range(ystart,yend+1,1)
alltimes = xr.concat(alltimes, dim='year')
alltimes['year'] = range(ystart,yend+1,1)

datout = xr.merge([alldat, alltimes])
datout.to_netcdf(outpath+'ERA5_monthly_'+ourvar+'_'+str(ystart)+'_'+str(yend)+'.nc')

1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025


In [47]:
cluster.close()